In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, median_absolute_error, mean_absolute_percentage_error
from sklearn.preprocessing import TargetEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

In [3]:
df = pd.read_csv("../data/processed/jakarta_properties_processed2.csv")

In [6]:
df.head()

,price_idr,district,bedrooms,bathrooms,garage,land_size_m2,building_size_m2,cluster,pool,mrt,tol,mall,city_Jakarta Barat,city_Jakarta Pusat,city_Jakarta Selatan,city_Jakarta Timur,city_Jakarta Utara
0,21.311053,pesanggrahan,3.0,2.0,2.0,4.795791,4.394449,0,0,0,0,0,0,0,1,0,0
1,22.654787,tebet,4.0,4.0,4.0,4.343805,5.929589,0,0,0,0,0,0,0,1,0,0
2,23.025851,kembangan,4.0,4.0,2.0,5.420535,6.216606,0,0,0,0,0,1,0,0,0,0
3,21.161521,kelapa gading,3.0,2.0,0.0,4.110874,4.110874,0,0,0,0,0,0,0,0,0,1
4,21.787977,kelapa gading,4.0,4.0,0.0,4.634729,4.634729,0,0,0,0,0,0,0,0,0,1


In [19]:
models_dict = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        objective="reg:absoluteerror",
        n_estimators=900,
        learning_rate=0.03,
        max_depth=13,
        subsample=0.8,
        colsample_bytree=0.7,
        min_child_weight=5,
        gamma=0.1,
        reg_alpha=1,
        reg_lambda=2,
        random_state=42
    )
}

In [20]:
X = df.drop(columns=["price_idr"])
y = df["price_idr"]

te = TargetEncoder(
    smooth=10,
    cv=5,
    target_type='continuous'
    )

X['district'] = te.fit_transform(X[['district']],y).flatten()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [21]:
def train_single_model(model_name,model,X_train,y_train,X_test,y_test):
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)

    result = {
        "Model" : model_name,
        "R2 Score":r2_score(y_test, y_pred),
        "MAE (mean)":mean_absolute_error(y_test, y_pred),
        "MDAE (median)": median_absolute_error(y_test, y_pred),
        "MAPE": mean_absolute_percentage_error(y_test, y_pred),
        "y_pred":y_pred
    }

    return result


In [24]:
def run_models(X_train, y_train, X_test, y_test, models):
    
    all_results = []

    for model_name, model in models.items():
        print(f"\nTraining {model_name} ...")

        result = train_single_model(
            model_name = model_name,
            model = model,
            X_train = X_train,
            y_train = y_train,
            X_test = X_test,
            y_test = y_test
        )

        all_results.append(result)

        print(f"Metrix Evaluation : {model_name}")
        print(f"R2 Score : {result["R2 Score"]}")
        print(f"MAE (mean): {result["MAE (mean)"]}")
        print(f"MDAE (median) : {result["MDAE (median)"]}")
        print(f"MAPE : {result["MAPE"]}")
        print(f"y_pred : {result["y_pred"]}")
        
        all_results_df = pd.DataFrame(all_results)
    return all_results_df

In [25]:
all_models_result = run_models(X_train = X_train, y_train = y_train, X_test = X_test, y_test = y_test, models = models_dict)
all_models_result


Training Linear Regression ...
Metrix Evaluation : Linear Regression
R2 Score : 0.8711906867179051
MAE (mean): 0.2774062194727147
MDAE (median) : 0.21362749703827788
MAPE : 0.012358598315778548
y_pred : [21.89355208 22.62477484 24.63059553 ... 22.54873369 22.67052494
 22.73828898]

Training Decision Tree ...
Metrix Evaluation : Decision Tree
R2 Score : 0.8698612665512356
MAE (mean): 0.2440575715432261
MDAE (median) : 0.15168205038049543
MAPE : 0.010896639932563719
y_pred : [21.78858474 22.7396852  24.35796612 ... 22.487912   22.81399523
 22.37860981]

Training Random Forest ...
Metrix Evaluation : Random Forest
R2 Score : 0.9068203234236561
MAE (mean): 0.21699741096462077
MDAE (median) : 0.15196768366462443
MAPE : 0.009674461682254361
y_pred : [21.71767191 22.68775133 24.42100499 ... 22.51306614 22.82619228
 22.78887969]

Training XGBoost ...
Metrix Evaluation : XGBoost
R2 Score : 0.9115904340565438
MAE (mean): 0.20072532673516744
MDAE (median) : 0.12971419299839404
MAPE : 0.008973573

,Model,R2 Score,MAE (mean),MDAE (median),MAPE,y_pred
0,Linear Regression,0.871191,0.277406,0.213627,0.012359,"[21.893552078997118, 22.624774840852034, 24.63..."
1,Decision Tree,0.869861,0.244058,0.151682,0.010897,"[21.788584744426025, 22.73968519575593, 24.357..."
2,Random Forest,0.906820,0.216997,0.151968,0.009674,"[21.717671907997378, 22.687751328389492, 24.42..."
3,XGBoost,0.911590,0.200725,0.129714,0.008974,"[21.760784, 22.668192, 24.339024, 21.82588, 23..."
